
# INTRODUCCIÓN
---

**MODELOS DE LENGUAJE Y PREDICCIÓN DE PALABRAS**


*Tipos de modelos de lenguaje*: Existen principalmente dos tipos de modelos de
lenguaje:

*   *Modelos de lenguaje estadístico:* estos modelos utilizan técnicas estadísticas tradicionales como N-gramas, modelos ocultos de Markov (HMM) y ciertas reglas lingüísticas para aprender la distribución de probabilidad de las palabras

*   *Modelos de lenguaje neuronal:* utilizan diferentes tipos de redes neuronales para modelar el lenguaje.

En este cuaderno abordaremos los modelos de lenguaje estadístico.

**INFERENCIA ESTADÍSTICA: modelos de n-gramas**

La inferencia estadística en general consiste en tomar algunos datos, generados según alguna distribución de probabilidad desconocida, y hacer algunas inferencias sobre esta distribución.


## MODELADO DE LENGUAJE

---

Un modelo de lenguaje aprende a predecir la probabilidad de una secuencia de palabras dado un conjunto de palabras previas. 

Esta tarea es fundamental para reconocimiento de voz, reconocimiento óptico de caracteres, corrección ortográfica, traducción automática, entre otras.

*   **PREDICCIÓN DE LA SIGUIENTE PALABRA:**
Lo que se intenta es calcular la función de probabilidad $P$

$$P(W_n / W_1, \ldots, W_{n-1})$$

Por ejemplo, en un modelo de bigramas buscaremos predecir la siguiente palabra a partir de la función de probabilidades de exactamente la palabra anterior.

Se utiliza la clasificaicón de las palabras previas, aunque debe considerarse que existen secuencias de palabras que no se encuentren en el historial, para obtener predicciones razonables dado este problema se agrupan segmentos históricos similares, utilizando la propiedad de Markov, que prioriza contextos locales.

Los modelos basados en n-gramas utilizan secuencias de 2,3,4  palabras consecutivas.

*   **ENFOQUE DE MAXIMA VEROSIMILITUD**

Para obtener las probabilidade que nos permitan estimar la probabilidad de aparición de una parabra objetivos, es necesario calcular

$$P(W_n | W_1, \ldots, W_{n-1}) = \frac{C(W_1, \ldots, W_{n-1}, W_n)}{C(W_1, \ldots, W_{n-1})}$$

Donde C(⋅) es el conteo de las secuencias de palabras de longitud $n$, por ejemplo, $n=2$ (dos palabras), $n=3$ (tres palabras), así sucesivamente.

Para calcular la probabilidad de una palabra $y$ dada una palabra previa $x$, se calcula el conteo de los bigramas  $C(xy)$ y se normaliza con todos los bigramas que comparten la primera palabra $x$, que es lo mismo que los unigramas de $x$


*   **SENSIBILIDAD DE LOS MODELOS DE N-GRAMAS Y SUAVIZADO**

Como otros modelos estadísticos, el modelo de n-gramas depende en gran medida  del conjunto de datos de entrenamiento. 

Lo que implica que las probabilidades codifican cuestiones específicas del conjunto de datos.

Para atenuar el problema de enfrentarse a probabilidades cero de n-gramas, se usa un suavizado de las probabilidades del conjunto de datos, esto es, se corta un poco de la masa de probabilidades de los conteos mayores, y se traslada a las secuencias que no tienen conteos.

**Suavizado de Laplace**

Suavizado de Laplace o ley de Laplace es una técnica sencilla que consiste en proporcionar un poco del espacio de probabilidades a los eventos no vistos. 

Esto es, se usa la matriz de conteo, por ejemplo de bigramas, y se suma 1 a todos los conteos, para posteriormente normalizar en probabilidades, bigramas que no ocurrieron en el conjunto de datos al menos tendrán una ocurrencia usando suavizado de Laplace. 

La probabilidad ajustada sería como sigue, donde se requiere ajustar agregando el tamaño del vocabulario V.
_________________________

In [ ]:
# Dependencies
import re
import pandas as pd

#
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# De texto a n-gramas

Texto = "¿Clases en sábado?"

Texto[1:8]

In [ ]:
#
Texto[ : 3]

In [ ]:
#
Texto[3 : ]

In [ ]:
#
Texto[ : -1]

In [ ]:
# Podemos utilizar esta técnica para tomar un número predeterminado de palabras vecinas 

cadenaPalabras = 'it was the best of times it was the worst of times, '

cadenaPalabras += 'it was the age of wisdom it was the age of foolishness'

cadenaPalabras

In [ ]:
# Texto

Texto = 'Clases lunes a viernes' \
        ' siempre' \
        'XXXoooXXX ' \
        '1 2 3'

Texto

In [ ]:
#

listaPalabras = cadenaPalabras.split()

In [ ]:
#

listaPalabras[ : ]

In [ ]:
#
listaPalabras[ 10 : ]

In [ ]:
#

listaPalabras[3:10]

In [ ]:
# Ejercicio....

listaPalabras[3][ 1:3 ]

#### Dada una lista de palabras y un número n, recupera una lista # de n-gramas.


In [ ]:
# Function:

def obtenNGramas(Lista_Palabras, n):
    return [ Lista_Palabras[ i : i + n ] for i in range( len( Lista_Palabras ) - ( n - 1 ) ) ]
# lista por comprensión para mantener el código compacto

In [ ]:
obtenNGramas(listaPalabras, 2)

In [ ]:
# Alternativamente:

def obtenNGramas(Lista_Palabras, n):
    ngramas = []
    for i in range( len(Lista_Palabras) - (n - 1) ):
        ngramas.append( Lista_Palabras[ i : i + n ])
    return ngramas

In [ ]:
obtenNGramas(listaPalabras, 2)

In [ ]:
listaPalabras

*** Utiliza el que tenga más sentido para ti. ***

In [ ]:
# Pongamos a trabajar nuestra función: obtenNGramas

Frase = 'it was the best of times it was the worst of times '
Frase += 'it was the age of wisdom it was the age of foolishness'

todasMisPalabras = cadenaPalabras.split()

todasMisPalabras

In [ ]:
#

NGRAMAS = obtenNGramas( todasMisPalabras, 2 )

NGRAMAS

In [ ]:
#[item for sublist in NGRAMAS for item in sublist]
#pd.Series([item for sublist in NGRAMAS for item in sublist]).value_counts()

In [ ]:
# Calculando frecuencias

grams_freq = pd.Series([item for sublist in NGRAMAS for item in sublist]).value_counts()

grams_freq

In [ ]:
# Calculando probabilidades

grams_freq_prob = grams_freq / grams_freq.sum()

grams_freq_prob

## Entrenamiento de un modelo N-gram

In [ ]:
from nltk.util import pad_sequence
from nltk.util import bigrams
from nltk.util import ngrams
from nltk.util import everygrams
from nltk.lm.preprocessing import pad_both_ends
from nltk.lm.preprocessing import flatten
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk import word_tokenize, sent_tokenize 
from nltk.tokenize import ToktokTokenizer
from nltk.lm import MLE

In [ ]:
# Abrir el archivo para lectura ('r' significa read)
with open('VEP_20230105_1.txt', 'r') as archivo:
    contenido = archivo.read()

# Ahora 'contenido' es una cadena de texto con todo el contenido del archivo
contenido

In [ ]:
# Preprocess the tokenized text for 3-grams language modelling

contenido = contenido.lower()

contenido

1. word_tokenize(contenido): Toma un texto (en la variable contenido) y lo divide (tokeniza) en palabras individuales, normalmente devolviendo una lista de palabras.

Ejemplo: Si contenido = "Hola mundo", entonces word_tokenize(contenido) devuelve:
$["Hola", "mundo"]$

2. pad_both_ends(..., n=2): Añade un relleno (“padding”) al inicio y al final de la lista de palabras. Parámetro: n=2 significa que se agregan dos tokens especiales al principio y dos al final. Usualmente se usan para construir n-gramas en procesamiento de lenguaje natural.

Ejemplo: Si la lista es $["Hola", "mundo"]$ y n=2, pad_both_ends($["Hola", "mundo"]$, n=2) devolvería algo como:
$["<s>", "<s>", "Hola", "mundo", "</s>", "</s>"]$
(Depende de cómo esté implementado, pero normalmente $<s>$ es inicio y $</s>$ es fin.)

3. list(...): Convierte el resultado en una lista. A veces, las funciones de padding devuelven un iterador o un generador, así que esto asegura que tienes una lista.

4. $[ ... ]$ (Lista que contiene una lista): El resultado del padding se pone dentro de otra lista. Es decir, tienes una lista que contiene una sola lista, por ejemplo: $[ ['<s>', '<s>', 'Hola', 'mundo', '</s>', '</s>'] ]$

Esto se hace muchas veces porque el código espera trabajar con varias líneas (cada línea sería una lista de palabras tokenizadas y “padeadas”), así que cada línea procesada es una lista y todas juntas están en una lista mayor.

In [ ]:
#

paddedLine = [ list( pad_both_ends( word_tokenize(contenido), n = 2) ) ]

paddedLine #paddedLine[0][-1], paddedLine[0][0] 

In [ ]:
# Su propósito principal es preparar datos para entrenar modelos de n-gramas (como el nltk.lm)
# Recibe como argumentos: El tamaño del n-grama (en este caso 5 → 5-gramas).
#                         Una secuencia de listas de tokens ya “padeadas” (es decir, ya con los <s> y </s>).

train, vocab = padded_everygram_pipeline(5, paddedLine)

# SEE: https://www.nltk.org/api/nltk.lm.preprocessing.html

padded_everygram_pipeline recibe el texto tokenizado y devuelve dos cosas:

a) train

* Es un generador (iterator) de everygrams, es decir, todas las secuencias posibles de hasta 5 palabras (1-gram, 2-gram… hasta 5-gram) en cada línea. Por ejemplo, para la frase anterior, te daría cosas como:	1-gramas: $<s>$, Hola, …, 2-gramas: $<s> <s>$, $<s>$ Hola, …, …, hasta los 5-gramas posibles. Usado para entrenar el modelo de lenguaje.

b) vocab

* Es otro generador, que produce todas las listas de palabras de tu corpus (en este caso, las palabras únicas de todas las frases). Sirve para crear el vocabulario del modelo.

¿Qué significa el 5? El 5 indica que quieres crear un modelo de 5-gramas. Es decir, el modelo mirará secuencias de hasta 5 palabras para aprender patrones y probabilidades.

In [ ]:
#

lm = MLE(5)

lm.fit(train, vocab)

In [ ]:
#

print(lm.counts)

In [ ]:
#

len(lm.vocab)

In [ ]:
#

lm.vocab.lookup('ley')

In [ ]:
#

lm.score("comisionado")

In [ ]:
# 

lm.counts['ley']

In [ ]:
#

lm.score("ley")

In [ ]:
#

lm.generate(1, random_seed = 3)

In [ ]:
#

lm.generate(5, random_seed = 3)

In [ ]:
#

lm.generate(20, text_seed = ['multar'], random_seed=3)